# 01 Connectivity inputs

Inspect config, selected recordings, source-label epoch inputs, and planned connectivity outputs.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif PROJECT_ROOT.name == "4_connectivity":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

CONFIG_PATH = PROJECT_ROOT / "configs" / "local.yaml"
print("PROJECT_ROOT:", PROJECT_ROOT)
print("CONFIG_PATH:", CONFIG_PATH)

from meeg_pipeline.config import load_config
from meeg_pipeline.workflow import iter_recordings, recordings_to_dataframe

config = load_config(CONFIG_PATH)
recordings = list(iter_recordings(config, subjects="all"))
recordings_to_dataframe(recordings)

In [ ]:
from meeg_pipeline.connectivity import (
    connectivity_config_to_dataframe,
    connectivity_input_overview_to_dataframe,
)

connectivity_config_to_dataframe(config)

In [ ]:
ON_EXISTING = "skip"  # "skip" | "overwrite" | "error"

overview = connectivity_input_overview_to_dataframe(
    config,
    recordings,
    on_existing=ON_EXISTING,
)
overview

In [ ]:
print("overview shape:", overview.shape)
print("overview columns:", list(overview.columns))

wanted_columns = [
    "subject", "task", "method", "window", "condition", "status",
    "n_epochs", "n_labels", "n_times", "selected_label_preview", "output_path",
]

available_columns = [col for col in wanted_columns if col in overview.columns]

if available_columns:
    display(overview[available_columns])
else:
    print("No expected overview columns available. The overview is probably empty.")


In [ ]:
# Quick check of expected selected labels for the first available source-label epoch file.
from pathlib import Path
import numpy as np
import pandas as pd

ready = overview[overview["ltc_path"].notna()] if "ltc_path" in overview.columns else overview
if ready.empty:
    raise RuntimeError("Connectivity overview is empty. Check that recordings were found and LTC epoch files exist.")

first = ready.iloc[0]
labels = pd.read_csv(first["labels_path"], sep="\t")
patterns = config.connectivity.label_patterns
if patterns:
    mask = False
    names = labels["label"].astype(str).str.lower()
    for pattern in patterns:
        mask = mask | names.str.contains(str(pattern).lower(), regex=False)
    selected_labels = labels.loc[mask].reset_index(drop=True)
else:
    selected_labels = labels

print("Selected labels:", len(selected_labels), "/", len(labels))
selected_labels.head(30)
